# Breast Cancer Transcriptomic Signature Analysis — GSE42568

This notebook uses a **public human transcriptomics dataset** from NCBI GEO (GSE42568). The biological source contains 104 breast-cancer biopsies and 17 normal breast-tissue samples.

## The biology in one minute
DNA contains genes. When a gene is active, the cell produces RNA from it. Transcriptomics measures many RNA-associated gene-expression signals at once. Cancer changes cell state, so some genes become more active and others less active.

Our question is: **Which expression signals differ reproducibly between breast tumor and normal breast tissue?**

## Why a public dataset matters
GSE42568 is deposited in the NCBI Gene Expression Omnibus, so another researcher can find the same accession and reproduce or challenge the analysis. The project uses a processed mirror on Zenodo for convenience, while the biological source remains GEO.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
print(ROOT)

## Step 1 — Run the complete reproducible analysis
The script downloads the data and platform annotation, checks the matrix, performs PCA, runs differential-expression tests, corrects for multiple testing, maps probes to genes, and creates figures/tables.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_analysis.py')], check=True)

## Step 2 — Inspect the results
**PCA:** compresses thousands of measurements into a few axes. If tumor and normal samples separate, that suggests broad transcriptomic differences.

**Welch's t-test:** compares average expression between tumor and normal tissue without assuming equal group variance.

**Benjamini–Hochberg FDR:** adjusts for the fact that we test thousands of features. A raw p-value alone would create many false positives.

**Volcano plot:** combines effect size (x-axis) with statistical evidence (y-axis).

**Heatmap:** shows the strongest gene-level signals across individual samples.

In [ ]:
import pandas as pd
summary = pd.read_csv(ROOT / 'results' / 'tables' / 'analysis_summary.csv', index_col=0)
summary

In [ ]:
gene_de = pd.read_csv(ROOT / 'results' / 'tables' / 'differential_expression_gene_level.csv')
gene_de[['gene_symbol','mean_tumor','mean_normal','mean_difference_tumor_minus_normal','fdr_bh']].head(20)

## Step 3 — Biological interpretation
Do **not** interpret a gene as a causal cancer gene merely because it is differentially expressed. Differential expression identifies association with the tumor state in this cohort. Biological interpretation should be supported by pathway analysis and external literature.

When you finish running the notebook, record:
1. Whether PCA separates tumor and normal samples.
2. How many features meet the chosen FDR/effect-size threshold.
3. The top mapped genes.
4. The strongest enriched pathways, if enrichment succeeds.
5. At least three limitations.

## Limitations to discuss
- This is a microarray dataset, not RNA-seq.
- Tumor and normal groups are imbalanced (many more tumors).
- Differential expression is associative, not causal.
- Tissue composition can affect measured expression.
- Probe-to-gene mapping can be many-to-one or ambiguous.
- Results should be validated in an independent cohort before claiming a robust biomarker.